# Quantum Federated Learning — walkthrough

Classifying points inside or outside the unit circle with a 2-qubit variational
circuit, trained in a *federated* setting: the data stays split across clients,
the server only ever aggregates weights.

This notebook is a **toy run** — 2 clients, 5 rounds, a couple hundred points,
under two minutes end to end — meant to show the mechanism and the code. The
actual thesis results come from server-side campaigns (7 seeds, 40 rounds,
simulated noise and CDR mitigation); those live in `README.md` as figures and
are regenerated with `python -m thesis_plots`.

In [ ]:
import os
import sys
from pathlib import Path

# Works both when launched from notebooks/ and from the repository root.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import matplotlib.pyplot as plt

from qibo_qfl_pt.task import (NLAYERS, NQUBITS, build_client_config,
                              create_model, create_trainable_layer,
                              evaluate_model, get_weights, load_data_client,
                              set_seed, set_weights, train_model)

plt.rcParams.update({"figure.figsize": (5, 3.4), "font.size": 9})
print(f"circuit: {NQUBITS} qubits, {NLAYERS} layers")

## The task and the data

Points drawn uniformly from $[-1, 1]^2$, labelled 1 when they fall inside the
unit circle. The boundary is curved, so the task is not linearly separable: the
model has to bend.

`load_data_client` is the same function the real clients call: it partitions the
dataset and hands back one client's shard. Two clients here, IID partitioning.

In [ ]:
N_CLIENTS = 2
NDATA = 120
SEED = 0

parts = [load_data_client(pid, ndata=NDATA, num_partitions=N_CLIENTS, seed=SEED)
         for pid in range(N_CLIENTS)]

fig, axes = plt.subplots(1, N_CLIENTS, figsize=(7, 3.2), constrained_layout=True)
theta = np.linspace(0, 2 * np.pi, 200)
for pid, (ax, (x, y)) in enumerate(zip(axes, parts)):
    x, y = x.numpy(), y.numpy()
    ax.scatter(*x[y == 1].T, s=12, color="C0", label="inside")
    ax.scatter(*x[y == 0].T, s=12, color="C3", label="outside")
    ax.plot(np.cos(theta), np.sin(theta), ls="--", lw=0.8, color="0.4")
    ax.set_title(f"Client {pid} — {int((y == 1).sum())}/{int((y == 0).sum())}")
    ax.set_aspect("equal")
    ax.set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")
axes[0].legend(frameon=False, loc="lower left", fontsize=8)
plt.show()

## The model

A 2-qubit variational circuit alternating, `NLAYERS` times, an *encoding* block
that writes the data point into rotation phases and a trainable block of RX/RZ
per qubit followed by a CNOT. The measured observable is $(Z_0 + Z_1)/2$, mapped
to $[0,1]$ and read as a class probability.

The model is a Python object built at runtime, not a config file. The diagram
below is the trainable part — the encoding gates are inserted at call time,
together with the data point.

In [ ]:
ansatz = create_trainable_layer()
for _ in range(NLAYERS - 1):
    ansatz = ansatz + create_trainable_layer()

print(ansatz.diagram())
print(f"\n{len(ansatz.get_parameters())} trainable parameters")

## Configuration

Everything that changes between runs lives in one dictionary. The keys match the
real pipeline's (`[tool.flwr.app.config]` in `pyproject.toml`), so the same
config can be handed straight to `build_client_config`.

In [ ]:
CONFIG = {
    "n_clients": 2,
    "n_rounds": 5,
    "strategy": "fedavg",
    "local_epochs": 1,
    "lr": 0.3,
    "ndata": NDATA,
    "seed": SEED,
    "mode": "noiseless",   # "noisy" / "mitigated" need a server, see the last section
    "nshots": "none",      # "none" = analytic expectation value, no sampling
}
CONFIG

## The federated loop

One round: the server sends out the current weights, each client trains locally
on its own shard, the server aggregates. The only thing that changes between
strategies is how it aggregates —

- **FedAvg**: average of the client weights, weighted by the number of examples.
- **FedAdam**: that same average becomes a *pseudo-gradient*
  $\Delta = \bar{w} - w$, and the server takes an Adam step along it. It damps
  the oscillations you get when clients pull in different directions.

> This is not the production pipeline: that one runs on Flower with Ray
> underneath, launched from `run_experiments/parallel_experiments.py`. Here the
> server-side aggregation is rewritten in twenty lines so you can see what it
> does, while the data, the model and the training are exactly the code in
> `qibo_qfl_pt/task.py`.

In [ ]:
def aggregate(client_weights, sizes, strategy, state,
              eta=0.1, beta1=0.9, beta2=0.99, tau=1e-3):
    """Aggregate client weights: FedAvg, or Adam on the pseudo-gradient."""
    total = sum(sizes)
    avg = [sum(w[i] * n for w, n in zip(client_weights, sizes)) / total
           for i in range(len(client_weights[0]))]
    if strategy == "fedavg":
        return avg

    delta = [a - g for a, g in zip(avg, state["global"])]
    state["m"] = [beta1 * m + (1 - beta1) * d for m, d in zip(state["m"], delta)]
    state["v"] = [beta2 * v + (1 - beta2) * d ** 2 for v, d in zip(state["v"], delta)]
    return [g + eta * m / (np.sqrt(v) + tau)
            for g, m, v in zip(state["global"], state["m"], state["v"])]


def run_federated(cfg, verbose=True):
    """Run the simulation and return the metrics round by round."""
    set_seed(cfg["seed"])
    x_test, y_test = load_data_client(0, ndata=cfg["ndata"],
                                      num_partitions=cfg["n_clients"],
                                      seed=cfg["seed"], client_eval=True)

    noise, mitigation, nshots = build_client_config(cfg, 0, 0)
    model = create_model("quantum", noise_model=noise, nshots=nshots,
                         mitigation_config=mitigation)
    global_w = get_weights(model)
    state = {"global": global_w,
             "m": [np.zeros_like(w) for w in global_w],
             "v": [np.zeros_like(w) for w in global_w]}

    loss, acc, f1 = evaluate_model(model, x_test, y_test)
    history = [{"round": 0, "loss": loss, "accuracy": acc, "f1": f1}]

    for rnd in range(1, cfg["n_rounds"] + 1):
        client_weights, sizes = [], []
        for pid in range(cfg["n_clients"]):
            xc, yc = load_data_client(pid, ndata=cfg["ndata"],
                                      num_partitions=cfg["n_clients"],
                                      seed=cfg["seed"])
            noise, mitigation, nshots = build_client_config(cfg, pid, rnd)
            client = create_model("quantum", noise_model=noise, nshots=nshots,
                                  mitigation_config=mitigation)
            set_weights(client, global_w)
            train_model(client, xc, yc, epochs=cfg["local_epochs"],
                        lr=cfg["lr"], verbose=False)
            client_weights.append(get_weights(client))
            sizes.append(len(xc))

        state["global"] = global_w
        global_w = aggregate(client_weights, sizes, cfg["strategy"], state)
        set_weights(model, global_w)

        loss, acc, f1 = evaluate_model(model, x_test, y_test)
        history.append({"round": rnd, "loss": loss, "accuracy": acc, "f1": f1})
        if verbose:
            print(f"  round {rnd}:  loss {loss:.4f}   accuracy {acc:.3f}")
    return history

## Run: FedAvg

Well under a minute.

In [ ]:
print("FedAvg")
history_avg = run_federated(CONFIG)

In [ ]:
def plot_history(histories, metric="loss"):
    fig, ax = plt.subplots(constrained_layout=True)
    for label, h in histories.items():
        rounds = [e["round"] for e in h]
        ax.plot(rounds, [e[metric] for e in h], marker="o", ms=4, label=label)
    ax.set_xlabel("Round $t$")
    ax.set_ylabel({"loss": "Loss", "accuracy": "Accuracy"}[metric])
    ax.legend(frameon=False)
    plt.show()


plot_history({"FedAvg": history_avg}, "loss")
plot_history({"FedAvg": history_avg}, "accuracy")

## Switching strategy

Change one string in the config; everything else stays the same.

A word on how to read the plot below: with five rounds and a single seed, the
comparison says nothing about which strategy is better. FedAdam starts slower
because the server-side moments need a few rounds to build up before they give a
useful step — which is exactly why the thesis curves run for 40 rounds across
seven seeds, with a MAD band around the median.

In [ ]:
print("FedAdam")
history_adam = run_federated({**CONFIG, "strategy": "fedadam"})

In [ ]:
histories = {"FedAvg": history_avg, "FedAdam": history_adam}
plot_history(histories, "loss")
plot_history(histories, "accuracy")

## What about the real results?

What you have seen is the mechanism, not a result: two clients, five rounds and
one seed tell you nothing about which strategy wins.

The thesis campaigns add everything missing here — 5 clients, 40 rounds, 7 seeds
reported as median and MAD, Pauli and readout noise drifting across rounds,
Clifford Data Regression mitigation with a memory of the map, a sweep over the
recalibration threshold — and they run on a server: a single noisy configuration
costs roughly 100 seconds per round against the 3 seconds here, because the
simulator switches to density matrices and starts sampling.

```bash
# federated campaign
python run_experiments/parallel_experiments.py --strategy FedAvg --workers 4

# centralized baseline
python run_experiments/centralized_experiments.py --mode mitigated --epochs 30

# figures and tables
python -m thesis_plots --list
python -m thesis_plots all
```